# Assignment 1: Forward Kinematics

Please install the required packages first, we need:
- numpy
- pandas
- matplotlib
- scipy
- pytest
- any autodiff package (e.g. jax, pytorch, tensorflow, ...)

In the following cells, you will find some code snipplets that help you to get started. 

## The musculoskeletal model structure

Technically, we only use a skeletal model here, but it is taken from the OpenSim musculoskeletal model found in `data/match_markers_but_ignore_physics.osim`. The model you are going to work with is found in `model/kintree.py`. It is a dictionary that describes the kinematic tree of the model. Each body has a parent body (except for the root body) and joints that connect it to its parent. Each joint has a type, an axis of rotation/translation, and limits.

The dictionary is structured in the following way:

```python
kintree = {
    "body_name": {
        "parent": "parent_body_name",
        "joints": {
            "joint_name": {
                "type": "hinge" or "slider",
                "axis": [x, y, z],
                "limits": [min, max]
            },
            ...
        },
        "children": {
            "child_body_name": {
                ...
            },
            ...
        }
    },
    ...
}
```
In the next cell, we use a recursive function to print the names of all bodies and their parent bodies in the kinematic tree.

In [ ]:
from model.kintree import kintree

# Iterate over the kintree and print the names of all bodies and their parent bodies
def print_kintree(kintree):
    for body_name, body_info in kintree.items():
        parent_name = body_info["parent"]
        print(f"Body: {body_name}, Parent: {parent_name}")
        # Recursively print children
        print_kintree(body_info.get("children", {}))
        
print_kintree(kintree)


Now, to find the joints connecting two bodies, you can access the joints connecting a body to its parent using the "joints" key in the dictionary:

In [ ]:
# In this example, we pick out the right femur body and print its joints
child_body = kintree['pelvis']['children']['femur_r']
for joint_name, joint_info in child_body['joints'].items():
    print(f"Joint: {joint_name}, Type: {joint_info['type']}, Axis: {joint_info['axis']}, Limits: {joint_info['limits']}")

You will see that the joint types are either "hinge" or "slider". A hinge joint allows rotation around a single axis, while a slider joint allows translation along a single axis.

## Data

All data for this exercise is stored in the `data/` folder. The marker data ends with `.trc` and the ground reaction forces, and kinematics, are stored in `.mot` files. You can use pandas functions to read these files. 
The files are tab-separated and have a few header lines that need to be skipped. `.trc` files have 4 header lines, `.mot` files have 5 or 10 header lines. `.trc` files use multicolumns, so you might need to do some additional processing to get the data in a nice format.

In [ ]:
# Loading
import pandas as pd
marker_data = pd.read_csv('data/markers.csv')
grf_data = pd.read_csv('data/grf.csv')
kinematics_data = pd.read_csv('data/angles_clean.csv')


In [ ]:
# Plotting the ground reaction forces, is the pattern as expected?
# Ground reaction forces naming convention:
# Left foot: ground_force_vx, ground_force_vy, ground_force_vz is the force vector
# Right foot: 1_ground_force_vx, 1_ground_force_vy, 1_ground_force_vz is the force vector
# _px, _py, _pz is the point of application of the force, and torque_x, _y, _z is the torque vector
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
plt.plot(grf_data['time'], grf_data['force_r_y'], label='Left Foot Fy')

## Is this pattern as expected?